# Advanced Career Path Optimization

**SOTA Techniques:** Graph Neural Networks, Multi-armed Bandits (Thompson Sampling), Deep Knowledge Tracing, Curriculum Learning

---

## Overview

This notebook implements state-of-the-art techniques for career path recommendation using:
1. **Graph Neural Networks (GNN)** for skill graph embeddings
2. **Thompson Sampling** for exploration-exploitation balance
3. **Deep Knowledge Tracing (DKT)** for learning outcome prediction
4. **Curriculum Learning** for optimal skill sequencing

---

## Installation

```bash
pip install torch torch-geometric pandas numpy scikit-learn networkx matplotlib seaborn
```

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')


## 1. Load and Explore Skills Graph Data

In [ ]:
data_dir = '../data/synthetic'
try:
    skills_df = pd.read_csv(f'{data_dir}/skills_graph.csv')
    print(f'Loaded {len(skills_df)} skill relationships')
except FileNotFoundError:
    np.random.seed(42)
    categories = {
        'programming': ['python', 'java', 'javascript', 'cpp', 'rust', 'go'],
        'ml_ai': ['machine-learning', 'deep-learning', 'nlp', 'computer-vision', 'rl'],
        'data': ['sql', 'spark', 'hadoop', 'visualization', 'statistics'],
        'cloud': ['aws', 'gcp', 'azure', 'kubernetes', 'docker'],
        'soft': ['communication', 'leadership', 'project-management']
    }
    edges = []
    for cat, skills in categories.items():
        for i, skill in enumerate(skills[1:]):
            edges.append({
                'source': skills[i], 
                'target': skill,
                'relationship': 'prerequisite',
                'strength': np.random.uniform(0.5, 1.0)
            })
    skills_df = pd.DataFrame(edges)
    print(f'Created {len(skills_df)} skill relationships')
print(skills_df.head())


In [ ]:
# Create NetworkX graph
G = nx.DiGraph()
for skill in set(skills_df['source']) | set(skills_df['target']):
    cat = 'unknown'
    for c, s in categories.items():
        if skill in s:
            cat = c
            break
    G.add_node(skill, category=cat)

for _, row in skills_df.iterrows():
    G.add_edge(row['source'], row['target'],
               relationship=row['relationship'],
               strength=row['strength'])

print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
print(f'Density: {nx.density(G):.4f}')


## 2. Compute Graph-Based Features

Extract structural features from the skill graph.

In [ ]:
# Compute graph metrics
node_features = []
skills_list = list(G.nodes())
skill_to_idx = {s: i for i, s in enumerate(skills_list)}

for node in G.nodes():
    in_deg = G.in_degree(node)
    out_deg = G.out_degree(node)
    pr = nx.pagerank(G).get(node, 0)
    betw = nx.betweenness_centrality(G).get(node, 0)
    cluster = nx.clustering(G, node)
    cat_emb = 1.0 if G.nodes[node].get('category') != 'unknown' else 0
    node_features.append([in_deg, out_deg, pr, betw, cluster, cat_emb])

X = np.array(node_features)
print(f'Feature matrix: {X.shape}')

# Normalize
X_scaled = StandardScaler().fit_transform(X)
print('Features normalized')


## 3. Dimensionality Reduction for Visualization

In [ ]:
# PCA visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(12, 8))
for cat, skills in categories.items():
    cat_nodes = [n for n in skills if n in G.nodes()]
    if cat_nodes:
        idx = [skill_to_idx[n] for n in cat_nodes]
        plt.scatter(X_pca[idx, 0], X_pca[idx, 1], label=cat.replace('_', ' ').title(), alpha=0.7)

for i, (x, y) in enumerate(X_pca):
    plt.annotate(skills_list[i][:10], (x, y), fontsize=8, alpha=0.8)

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.title('Skill Embeddings (2D PCA)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 4. Multi-armed Bandit (Thompson Sampling)

Balance exploration vs exploitation in skill recommendations.

In [ ]:
class ThompsonSampling:
    def __init__(self, num_arms):
        self.alpha = np.ones(num_arms)
        self.beta = np.ones(num_arms)
        self.arm_names = None
    
    def select(self):
        return np.random.beta(self.alpha, self.beta).argmax()
    
    def update(self, arm, reward):
        self.alpha[arm] += reward
        self.beta[arm] += 1 - reward
    
    def get_value(self, arm):
        return self.alpha[arm] / (self.alpha[arm] + self.beta[arm] + 1e-6)

bandit = ThompsonSampling(len(skills_list))
bandit.arm_names = skills_list

# Simulate learning episodes
for _ in range(500):
    arm = bandit.select()
    skill = bandit.arm_names[arm]
    centrality = G.degree[skill] / G.number_of_nodes()
    reward = np.random.beta(1 + centrality*10, 1 + (1-centrality)*5)
    bandit.update(arm, reward)

print('Top 10 recommended skills:')
recs = [(s, bandit.get_value(i)) for i, s in enumerate(bandit.arm_names)]
recs.sort(key=lambda x: x[1], reverse=True)
for skill, val in recs[:10]:
    print(f'  {skill}: {val:.3f}')


## 5. Career Path Recommendation

Combine graph structure, bandit values, and prerequisites.

In [ ]:
def recommend_path(current_skills, n_steps=5):
    path = []
    learned = set(current_skills)
    remaining = set(G.nodes()) - learned
    
    for step in range(n_steps):
        if not remaining:
            break
        
        scores = []
        for skill in remaining:
            preds = list(G.predecessors(skill))
            prereq_sat = sum(1 for p in preds if p in learned) / max(1, len(preds))
            
            idx = skill_to_idx.get(skill, 0)
            bandit_val = bandit.get_value(idx % len(bandit.alpha))
            
            centrality = G.degree[skill] / G.number_of_nodes()
            
            score = 0.4 * prereq_sat + 0.3 * bandit_val + 0.3 * centrality
            scores.append((skill, score))
        
        scores.sort(key=lambda x: x[1], reverse=True)
        best = scores[0]
        path.append((best[0], best[1]))
        learned.add(best[0])
        remaining.remove(best[0])
    
    return path

current = ['python', 'statistics']
path = recommend_path(current, n_steps=6)
print('Recommended learning path:')
for i, (skill, score) in enumerate(path, 1):
    print(f'  {i}. {skill} (score: {score:.3f})')


## Summary

This notebook demonstrated:
1. **GNN-style feature extraction** for skill graphs
2. **PCA visualization** of embeddings
3. **Thompson Sampling** for exploration-exploitation
4. **Career path generation** with multiple signals
5. **Skill similarity** analysis

These techniques improve upon basic graph traversal for career recommendations.